## AirBnB Rio - Data Analytics Project

In [1]:
# Importing necessary libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

Refs:
https://code.visualstudio.com/docs/python/jupyter-support-py
https://code.visualstudio.com/docs/datascience/jupyter-notebooks

### Extraction

Our first order of business is matching our public safety dataset with the AirBnB dataset.
The dificulty lies in the fact that the latter is organized by neighbourhood whilst the first is organized by 'territory units' ('Unidade Territorial') which include one or more neighbourhoods.
Fortunately, the following table will allow us to translate those territorial units into neighbourhoods and match the two datasets.

In [3]:
# Loading the Public Security dataset
RAC = pd.read_csv("Relacao_RISPxAISPxCISP_edit.csv")
RAC.head(15)
#RAC.rename(columns={'Unidade Territorial': 'neighbourhood'}, inplace=True)

,RISP,AISP,CISP,Unidade Territorial,Município,Região de Governo
0,1,2,9,"Catete, Cosme Velho, Flamengo, Glória e Laranj...",Rio de Janeiro,Metropolitana
1,1,2,10,"Botafogo, Humaitá e Urca",Rio de Janeiro,Metropolitana
2,1,3,23,"Cachambi, Méier (parte) e Todos os Santos (parte)",Rio de Janeiro,Metropolitana
3,1,3,24,"Abolição, Água Santa (parte), Encantado, Engen...",Rio de Janeiro,Metropolitana
4,1,3,25,"Engenho Novo, Jacaré, Jacarezinho, Riachuelo, ...",Rio de Janeiro,Metropolitana
5,1,3,26,"Água Santa (parte), Engenho de Dentro (parte),...",Rio de Janeiro,Metropolitana
6,1,3,44,"Del Castilho, Engenho da Rainha, Inhaúma, Mari...",Rio de Janeiro,Metropolitana
7,1,4,6,"Catumbi, Cidade Nova, Estácio, Rio Comprido e ...",Rio de Janeiro,Metropolitana
8,1,4,17,"Caju, Mangueira, São Cristóvão e Vasco da Gama",Rio de Janeiro,Metropolitana
9,1,5,1,Centro (parte),Rio de Janeiro,Metropolitana


First, we identify every unique neighbourhood in the dataset.

In [4]:
## A little script for creating an individual neighbourhood *array*

# Function for flattening a list
def flat(lis):
    flatList = []
    # Iterate with outer list
    for element in lis:
        if type(element) is list:
            # Check if type is list than iterate through the sublist
            for item in element:
                flatList.append(item)
        else:
            flatList.append(element)
    return flatList

# Creating the individual neighbourhood array
neighbourhoods = []
for UT in RAC["Unidade Territorial"]:
    UT = UT.replace(" (parte)","").replace(" e ",",")
    nb = UT.split(",")
    neighbourhoods.append(nb)

neighbourhoods = flat(neighbourhoods)
neighbourhoods = [nb.strip() for nb in neighbourhoods]
print(neighbourhoods)
#neighbourhoods = neighbourhoods.drop_duplicates()

['Catete', 'Cosme Velho', 'Flamengo', 'Glória', 'Laranjeiras', 'Botafogo', 'Humaitá', 'Urca', 'Cachambi', 'Méier', 'Todos os Santos', 'Abolição', 'Água Santa', 'Encantado', 'Engenho de Dentro', 'Pilares', 'Piedade', 'Engenho Novo', 'Jacaré', 'Jacarezinho', 'Riachuelo', 'Rocha', 'Sampaio', 'São Francisco Xavier', 'Água Santa', 'Engenho de Dentro', 'Lins de Vasconcelos', 'Todos os Santos', 'Del Castilho', 'Engenho da Rainha', 'Inhaúma', 'Maria da Graça', 'Tomás Coelho', 'Catumbi', 'Cidade Nova', 'Estácio', 'Rio Comprido', 'Centro', 'Caju', 'Mangueira', 'São Cristóvão', 'Vasco da Gama', 'Centro', 'Centro', 'Gamboa', 'Santo Cristo', 'Saúde', 'Centro', 'Lapa', 'Paquetá', 'Santa Teresa', 'Maracanã', 'Praça da Bandeira', 'Tijuca', 'Alto da Boa Vista', 'Tijuca', 'Andaraí', 'Grajaú', 'Vila Isabel', 'Brás de Pina', 'Olaria', 'Penha', 'Penha Circular', 'Brás de Pina', 'Cordovil', 'Jardim América', 'Parada de Lucas', 'Penha Circular', 'Vigário Geral', 'Bancários', 'Cacuia', 'Cidade Universitária',

We will then use the array "neighbourhoods" and the original table above to assign the police districts (RISP, AISP and CISP) to every neighbourhood - instead of territory unit.

In [5]:
# I will transform the data array 'neighbourhoods' into a dataframe and merge it with the public security dataframe, so that each listing has its respective AISP, RISP and CISP.
df_neigh = pd.DataFrame({"Neighbourhood": neighbourhoods})

df_neigh["RISP"] = None
df_neigh["AISP"] = None
df_neigh["CISP"] = None

# Loop — searches for the neighbourhood inside the 'Unidade Territorial' string
for i, row_n in df_neigh.iterrows():
    bairro = row_n["Neighbourhood"]
    for j, row_c in RAC.iterrows():
        if bairro.lower() in row_c["Unidade Territorial"].lower():
            df_neigh.at[i, "RISP"] = row_c["RISP"]
            df_neigh.at[i, "AISP"] = row_c["AISP"]
            df_neigh.at[i, "CISP"] = row_c["CISP"]
            break  # stops as soon as it finds the first match

print(df_neigh)

       Neighbourhood RISP AISP CISP
0             Catete    1    2    9
1        Cosme Velho    1    2    9
2           Flamengo    1    2    9
3             Glória    1    2    9
4        Laranjeiras    1    2    9
..               ...  ...  ...  ...
168            Acari    2   41   39
169     Barros Filho    2   41   39
170     Costa Barros    2   41   39
171  Parque Colúmbia    2   41   39
172           Pavuna    2   41   39

[173 rows x 4 columns]


In [6]:
print(df_neigh.value_counts())
# There seems to be a few duplicated rows
df_neigh[df_neigh.duplicated(subset="Neighbourhood", keep=False)]
# At least they don't differ in values, so we can just delete the extra ones
df_neigh = df_neigh.drop_duplicates(subset="Neighbourhood", keep="first")

print(df_neigh.value_counts())
print(df_neigh[df_neigh.duplicated(subset="Neighbourhood", keep=False)])

Neighbourhood  RISP  AISP  CISP
Centro         1     4     6       4
Água Santa     1     3     24      2
Oswaldo Cruz   2     9     29      2
Copacabana     1     19    12      2
Colégio        2     9     40      2
                                  ..
Grumari        2     31    42      1
Guadalupe      2     41    31      1
Guaratiba      2     27    43      1
Gávea          1     23    15      1
Laranjeiras    1     2     9       1
Name: count, Length: 160, dtype: int64
Neighbourhood       RISP  AISP  CISP
Abolição            1     3     24      1
Acari               2     41    39      1
Pavuna              2     41    39      1
Pechincha           2     18    41      1
Pedra de Guaratiba  2     27    43      1
                                       ..
Gericinó            2     14    34      1
Glória              1     2     9       1
Grajaú              1     6     20      1
Grumari             2     31    42      1
Água Santa          1     3     24      1
Name: count, Length: 16

Before moving onward, let's add 'neighbourhood groups' to the dataframe, as there is a column in the AirBnb dataset for it but no data.

In [15]:
# Neighbourhoods Groups
zona_central = {
    "Centro", "Lapa", "Gamboa", "Santo Cristo", "Saúde",
    "Catumbi", "Cidade Nova", "Estácio", "Rio Comprido",
    "Caju", "Mangueira", "São Cristóvão", "Vasco da Gama",
    "Paquetá", "Santa Teresa", "Glória"
}

zona_sul = {
    "Catete", "Flamengo", "Laranjeiras", "Botafogo", "Humaitá",
    "Urca", "Copacabana", "Leme", "Ipanema", "Leblon",
    "Gávea", "Jardim Botânico", "Lagoa", "São Conrado",
    "Vidigal", "Rocinha",
    "Tijuca", "Alto da Boa Vista", "Andaraí", "Grajaú",
    "Vila Isabel", "Maracanã", "Praça da Bandeira",
    "Cosme Velho"
}

zona_norte = {
    "Méier", "Todos os Santos", "Abolição", "Água Santa",
    "Encantado", "Engenho de Dentro", "Pilares", "Piedade",
    "Engenho Novo", "Jacaré", "Jacarezinho", "Riachuelo",
    "Rocha", "Sampaio", "São Francisco Xavier",
    "Lins de Vasconcelos", "Del Castilho", "Engenho da Rainha",
    "Inhaúma", "Maria da Graça", "Tomás Coelho",
    "Benfica", "Bonsucesso", "Higienópolis", "Manguinhos",
    "Maré", "Ramos",
    "Brás de Pina", "Olaria", "Penha", "Penha Circular",
    "Cordovil", "Jardim América", "Parada de Lucas",
    "Vigário Geral",
    "Ilha do Governador", "Bancários", "Cacuia",
    "Cidade Universitária", "Cocotá", "Freguesia",
    "Galeão", "Jardim Carioca", "Jardim Guanabara",
    "Moneró", "Pitangueiras", "Portuguesa",
    "Praia da Bandeira", "Ribeira", "Tauá", "Zumbi",
    "Madureira", "Cavalcanti", "Engenheiro Leal",
    "Turiaçu", "Vaz Lobo", "Oswaldo Cruz", "Cascadura",
    "Quintino Bocaiúva", "Bento Ribeiro", "Campinho",
    "Marechal Hermes",
    "Coelho Neto", "Colégio", "Honório Gurgel",
    "Rocha Miranda",
    "Irajá", "Vicente de Carvalho", "Vila Kosmos",
    "Vila da Penha", "Vista Alegre",
    "Anchieta", "Guadalupe", "Parque Anchieta",
    "Ricardo de Albuquerque",
    "Acari", "Barros Filho", "Costa Barros",
    "Parque Colúmbia", "Pavuna", "Cachambi"
}

zona_sudoeste = {
    "Barra da Tijuca", "Itanhangá", "Joá",
    "Recreio dos Bandeirantes", "Camorim", "Grumari",
    "Vargem Grande", "Vargem Pequena",
    "Jacarepaguá", "Taquara", "Anil", "Cidade de Deus",
    "Curicica", "Gardênia Azul",
    "Freguesia (Jacarepaguá)", "Pechincha", "Tanque"
}

zona_oeste = {
    "Campo dos Afonsos", "Deodoro", "Jardim Sulacap",
    "Magalhães Bastos", "Realengo", "Vila Militar",
    "Bangu", "Gericinó", "Padre Miguel",
    "Senador Camará", "Vila Valqueire", "Praça Seca",
    "Paciência", "Santa Cruz",
    "Guaratiba", "Pedra de Guaratiba",
    "Sepetiba", "Barra de Guaratiba",
    "Campo Grande", "Cosmos", "Inhoaíba",
    "Santíssimo", "Senador Vasconcelos"
}

In [ ]:
def classify_zone(neighbourhood):
    if neighbourhood in zona_central:
        return "Zona Central"
    elif neighbourhood in zona_sul:
        return "Zona Sul"
    elif neighbourhood in zona_norte:
        return "Zona Norte"
    elif neighbourhood in zona_sudoeste:
        return "Zona Sudoeste"
    elif neighbourhood in zona_oeste:
        return "Zona Oeste"
    else:
        return "Unclassified"

# Creating a new column

df_neigh["neighbourhood_group"] = df_neigh["Neighbourhood"].apply(classify_zone)
#df_neigh[df_neigh["neighbourhood_group"] == "Unclassified"]
df_neigh.head(15)

,Neighbourhood,RISP,AISP,CISP,neighbourhood_group
0,Catete,1,2,9,Zona Sul
1,Cosme Velho,1,2,9,Zona Sul
2,Flamengo,1,2,9,Zona Sul
3,Glória,1,2,9,Zona Central
4,Laranjeiras,1,2,9,Zona Sul
5,Botafogo,1,2,10,Zona Sul
6,Humaitá,1,2,10,Zona Sul
7,Urca,1,2,10,Zona Sul
8,Cachambi,1,3,23,Zona Norte
9,Méier,1,3,23,Zona Norte


In [17]:
df_neigh.to_csv("Neighbourhood_CISP.csv", index=False)